# P8-1 · Bài 10a · Serving

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q fastapi openai

## Lý thuyết

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Review sentiment API")

class Req(BaseModel):
    text: str = Field(min_length=1, max_length=2000)

class Resp(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

def model_predict(text):                         # thay cho model thật: luật đơn giản để chạy thử
    pos = any(w in text.lower() for w in ("tốt", "đẹp", "thích", "nhanh"))
    return ("tích cực", 0.9) if pos else ("tiêu cực", 0.6)

@app.post("/predict", response_model=Resp)
def predict(req: Req):
    label, score = model_predict(req.text)       # model nạp một lần khi khởi động
    return Resp(label=label, score=score)

# gọi thử API ngay trong notebook, không cần chạy uvicorn
from fastapi.testclient import TestClient
api = TestClient(app)
print(api.get("/health").json())
print(api.post("/predict", json={"text": "Ghế đẹp, giao nhanh"}).json())
r = api.post("/predict", json={"text": ""})
print(r.status_code, r.json()["detail"][0]["msg"])             # Pydantic từ chối: 422

{'status': 'ok'}
{'label': 'tích cực', 'score': 0.9}
422 String should have at least 1 character


In [ ]:
import asyncio
from fastapi.responses import StreamingResponse

async def fake_llm(text, n=150):                 # giả lập LLM 50 token/giây
    for i in range(n):
        await asyncio.sleep(0.02)
        yield f"tok{i} "

@app.post("/chat/stream")
async def chat_stream(req: Req):
    async def gen():
        async for piece in fake_llm(req.text):
            yield f"data: {piece}\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

# chạy server thật (uvicorn) ở luồng nền rồi đo thời gian tới mảnh đầu tiên và tổng thời gian
import threading, time, httpx, uvicorn
server = uvicorn.Server(uvicorn.Config(app, port=8899, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.05)

t0, first, n = time.perf_counter(), None, 0
with httpx.stream("POST", "http://127.0.0.1:8899/chat/stream", json={"text": "xin chào"}) as resp:
    for line in resp.iter_lines():
        if line.startswith("data:"):
            n += 1
            first = first or time.perf_counter() - t0
print(f"{n} mảnh | mảnh đầu sau {first * 1000:.0f} ms | xong sau {time.perf_counter() - t0:.2f} s")
server.should_exit = True

151 mảnh | mảnh đầu sau 52 ms | xong sau 3.08 s


## Bài tập 4: API có streaming và kiểm tra đầu vào {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần vLLM server đang chạy ở cổng 8000 (bước 2 của phần thực hành)
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import OpenAI, APIError

app = FastAPI()
client = OpenAI(base_url="http://localhost:8000/v1", api_key="local", timeout=30)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

class ChatReq(BaseModel):
    message: str = Field(min_length=1, max_length=2000)

@app.get("/health")
def health():
    try:
        client.models.list()
        return {"status": "ok"}
    except Exception:
        raise HTTPException(503, "model chưa sẵn sàng")

@app.post("/chat")
def chat(req: ChatReq):
    try:
        stream = client.chat.completions.create(model=MODEL, stream=True, max_tokens=512,
                                                messages=[{"role": "user", "content": req.message}])
    except APIError as e:
        raise HTTPException(503, f"lỗi model: {e}")

    def gen():
        for chunk in stream:
            delta = chunk.choices[0].delta.content if chunk.choices else None
            if delta:
                yield f"data: {delta}\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

## Bài tập 5: Nạp model một lần khi khởi động {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import time
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
so_lan_nap = {"sai": 0, "dung": 0}

def nap_model(ban):
    so_lan_nap[ban] += 1
    time.sleep(0.2)                                     # đọc trọng số từ đĩa, chuyển lên GPU...
    return lambda text: {"nhan": "tích cực" if "tốt" in text else "tiêu cực"}

class Req(BaseModel):
    text: str = Field(min_length=1, max_length=500)
# TODO: app_sai (nạp trong handler), app_dung (nạp trong lifespan, có /health); gửi 5 request mỗi app

In [ ]:
app_sai = FastAPI()

@app_sai.post("/predict")
def predict_sai(req: Req):
    model = nap_model("sai")
    return model(req.text)

@asynccontextmanager
async def lifespan(app):
    app.state.model = nap_model("dung")
    yield

app_dung = FastAPI(lifespan=lifespan)

@app_dung.get("/health")
def health():
    return {"ok": True}

@app_dung.post("/predict")
def predict_dung(req: Req):
    return app_dung.state.model(req.text)

for ten, app in [("sai", app_sai), ("dung", app_dung)]:
    with TestClient(app) as c:
        t0 = time.perf_counter()
        for _ in range(5):
            r = c.post("/predict", json={"text": "bàn tốt lắm"})
        print(f"bản {ten}: {so_lan_nap[ten]} lần nạp, {time.perf_counter() - t0:.2f} s cho 5 request, kết quả {r.json()}")
        ma_rong = c.post("/predict", json={"text": ""}).status_code
print("text rỗng →", ma_rong)

bản sai: 5 lần nạp, 1.01 s cho 5 request, kết quả {'nhan': 'tích cực'}
bản dung: 1 lần nạp, 0.01 s cho 5 request, kết quả {'nhan': 'tích cực'}
text rỗng → 422


## Bài tập 6: Bảng tính năng lực và chi phí {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
kich_ban = {"chatbot nội bộ": (2, 4.0, 16, 1.2), "chatbot khách hàng": (40, 3.0, 16, 1.2), "tạo ảnh": (5, 6.0, 1, 0.8)}
#                                  (request/giây, giây mỗi request, request đồng thời 1 GPU chịu được, USD/giờ/GPU)

def uoc_tinh(rps, giay, dong_thoi_gpu, gia):
    # TODO: trả về (đồng thời, số GPU, USD mỗi 1.000 request)
    return 0, 0, 0.0

In [ ]:
def uoc_tinh(rps, giay, dong_thoi_gpu, gia):
    L = rps * giay
    gpu = math.ceil(L * 1.3 / dong_thoi_gpu)
    return L, gpu, gpu * gia / (rps * 3600) * 1000

print(f"{'kịch bản':20s} {'đồng thời':>9} {'GPU':>4} {'USD/1000 req':>13}")
for ten, ts in kich_ban.items():
    L, g, c = uoc_tinh(*ts)
    print(f"{ten:20s} {L:9.0f} {g:4d} {c:13.3f}")

kịch bản             đồng thời  GPU  USD/1000 req
chatbot nội bộ               8    1         0.167
chatbot khách hàng         120   10         0.083
tạo ảnh                     30   39         1.733
